
# 04 — NUTS vs. Fixed-Step Leapfrog: When the Fancier Tool Loses

**Prerequisite:** Notebook `00` (leapfrog basics) is essential background here.

Recall from Notebook `00`: the leapfrog integrator needs two hand-tuned
numbers — **step size** ($\varepsilon$) and **trajectory length** ($L$, how
many leapfrog steps to take per proposal). The project's Method A and Method
C use fixed, hand-chosen values for both.

The **No-U-Turn Sampler (NUTS)** is a well-known, more sophisticated
alternative that removes both of these by hand-tuning:
- **Step size** is adapted automatically via a technique called *dual
  averaging*, which nudges $\varepsilon$ up or down after each proposal to
  hit a target acceptance rate.
- **Trajectory length** is chosen per-proposal by *doubling a binary tree* of
  leapfrog steps — imagine walking away from home, doubling your distance
  each time you check, and stopping the moment you notice you've started
  curving back toward where you began (the "no U-turn" moment).

This project built NUTS from scratch and tested it head-to-head against
fixed-step leapfrog. **NUTS lost.** This notebook shows you the live
mechanism, the real 5-seed result, and the concrete, checkable reason why.


In [ ]:

import sys, os, json
import numpy as np

REPO_PATH = None
_candidates = [REPO_PATH, "HPO-HMC", os.path.join("..", "HPO-HMC"), "."]
for _c in _candidates:
    if _c and os.path.isdir(os.path.join(_c, "src")):
        REPO_PATH = _c
        break
if REPO_PATH is None:
    raise FileNotFoundError("Couldn't find the HPO-HMC repo. Set REPO_PATH manually.")
print(f"Using repo at: {os.path.abspath(REPO_PATH)}")

sys.path.insert(0, os.path.join(REPO_PATH, "src"))
sys.path.insert(0, REPO_PATH)

import torch
import torch.nn as nn

from hamiltonian import HamiltonianNN, HyperparamState
from data_generator import generate_hamiltonian_data
from symplectic_solver import compute_loss_and_grads
from nuts_integrator import NUTSSampler
import config as base_config

torch.manual_seed(0)
np.random.seed(0)
train_loader, val_loader, _ = generate_hamiltonian_data(n_samples=500, seed=0)
criterion = nn.MSELoss()



## 1. NUTS live: watch the step size adapt and the tree grow

Below, `sampler.eps` is the automatically-adapted step size (compare this to
Method A/C's fixed `step_size=0.005` everywhere else in this project), and
`sampler.last_tree_depth` / `last_n_leapfrog` show how long NUTS decided each
individual proposal's trajectory needed to be.


In [ ]:

hp_state = HyperparamState(base_config.INIT_HYPERPARAMS, base_config.HYPERPARAM_SPACE)
hp_state.frozen_hps = ["n_layers", "n_neurons"]
hp = hp_state.decode()
model = HamiltonianNN(n_layers=hp["n_layers"], n_neurons=hp["n_neurons"],
                       dropout=hp["dropout"], input_dim=2)

sampler = NUTSSampler(step_size=0.005, mass_theta=1.0, mass_lambda=base_config.MASS_LAMBDA,
                       max_tree_depth=5, target_accept=0.8, n_adapt=5)

Xb, yb = next(iter(train_loader))
loss, _ = compute_loss_and_grads(model, (Xb, yb), criterion)

print(f"{'step':>4} {'loss':>10} {'adapted eps':>12} {'tree depth':>11} {'leapfrog steps used':>20}")
for step in range(8):
    accepted, loss = sampler.propose(model, hp_state, (Xb, yb), criterion, loss)
    print(f"{step:>4} {loss:>10.4f} {sampler.eps:>12.5f} {sampler.last_tree_depth:>11d} {sampler.last_n_leapfrog:>20d}")

print(f"\nNotice the step size (eps) changes each iteration during adaptation,")
print(f"then freezes once n_adapt steps are done -- that's dual averaging at work.")
print(f"Notice the tree depth often hits (or nearly hits) max_tree_depth={sampler.max_tree_depth},")
print(f"meaning it used up to {2**sampler.max_tree_depth - 1} leapfrog steps for ONE proposal --")
print(f"compare that to Method A/C's fixed 6 steps per proposal, always.")



## 2. The real, full 5-seed comparison

The live demo above is illustrative but tiny. Here's what the project's full
head-to-head comparison found (harmonic oscillator, 5 seeds, identical
Adam-warmup/L-BFGS curriculum on both arms, both samplers run as genuine MCMC
so the comparison isolates the integrator itself — not the "always-accept
optimisation mode" trick Method C normally uses, which NUTS has no equivalent
of).


In [ ]:

summary = json.load(open(os.path.join(REPO_PATH, "results", "nuts_comparison", "summary.json")))

print(f"{'Sampler':<12} {'Best Val. MSE':>18} {'Wall time (s)':>16} {'Steps/proposal':>16} {'Accept rate':>13}")
for name, key in [("Leapfrog", "leapfrog"), ("NUTS", "nuts")]:
    s = summary[key]
    print(f"{name:<12} {s['mse_mean']:>10.3f}+/-{s['mse_std']:<6.3f} "
          f"{s['time_mean']:>14.1f} {s['mean_leapfrog_per_proposal']:>16.1f} "
          f"{s['acceptance_rate_mean']:>12.1%}")

from scipy import stats
results = json.load(open(os.path.join(REPO_PATH, "results", "nuts_comparison", "results.json")))
seeds = sorted(set(r["seed"] for r in results))
lf = [next(r["best_val_mse"] for r in results if r["sampler"]=="leapfrog" and r["seed"]==s) for s in seeds]
nu = [next(r["best_val_mse"] for r in results if r["sampler"]=="nuts" and r["seed"]==s) for s in seeds]
w_stat, w_p = stats.wilcoxon(lf, nu)
print(f"\nPaired Wilcoxon signed-rank test (5 matched seeds): p = {w_p:.3f}")
print("(not significant at this sample size, but directionally consistent --")
print(" leapfrog had the lower MSE on 4 of the 5 seeds)")



## 3. Why did NUTS lose?

Look at the "steps/proposal" column above: fixed-step leapfrog always uses
exactly 6 steps. NUTS's tree-doubling grew to a **mean of ~61.6 steps** —
almost always hitting the maximum allowed depth (63) instead of detecting a
U-turn early. That means **every NUTS proposal cost roughly 10x what a
leapfrog proposal cost**, for a numerically *worse* (though not statistically
significant) result.

**The likely cause:** a properly-tuned, production-grade NUTS implementation
(like Stan's) adapts a separate "mass" scale for each part of the system it's
exploring. This implementation didn't — so the neural network's weights
(mass $m_\theta = 1$) and the hyperparameters (mass $m_\lambda \approx 0.1$)
were being explored with a one-size-fits-all yardstick. When two parts of a
system live on very different natural scales, an isotropic (one-size-fits-all)
no-U-turn check struggles to detect a turn quickly in either subspace, so the
tree keeps doubling without ever triggering the stopping condition.

### What this teaches you

A more sophisticated method isn't automatically a better one — it's better
*only if you give it what it needs to work* (here: mass-matrix adaptation).
Reporting this as a clean, explained loss — rather than quietly dropping the
experiment, or worse, tuning it until NUTS happened to win — turns a "we
tried something and it didn't help" moment into a concrete, useful, and
honest piece of engineering knowledge.

**Next notebook (`05`):** the whole project, all its numbers, compressed into
one place — plus what any of it might mean for you next.
